In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
import intrasom

# Settings
pd.set_option('display.max_columns', None)

In [2]:
# ----------- Basisverzeichnis (aktuelles Notebook-Verzeichnis)
base_dir = Path.cwd()
# Da wir jetzt im Unterordner '3.2_Machine-Learning' sind, ist Preprocessing in '3.1_Preprocessing/Preprocessing'
preprocessing_root = base_dir.parent / "3.1_Preprocessing" / "Preprocessing"

# ----------- Suche nach dem neusten Preprocessing-Ordner
timestamp_folders = [f for f in preprocessing_root.iterdir() if f.is_dir()]
if not timestamp_folders:
    raise FileNotFoundError(f"Keine Preprocessing-Ordner in {preprocessing_root} gefunden.")

latest_folder = max(timestamp_folders, key=lambda f: f.stat().st_mtime)
print(f"Verwendeter Preprocessing-Ordner: {latest_folder.name}")

# Parsing des Preprocessing-Zeitstempels aus dem Ordnernamen
try:
    prep_ts = datetime.strptime(latest_folder.name, "%Y-%m-%d_%H-%M-%S")
except ValueError:
    print("Warnung: Preprocessing-Ordnername entspricht nicht dem Schema. Nutze Dateisystem-Zeit.")
    prep_ts = datetime.fromtimestamp(latest_folder.stat().st_mtime)

# ----------- Lade Preprocessed Data
input_path_prep = latest_folder / "Preprocessed_SOM_Ready.csv"
df_prep = pd.read_csv(input_path_prep, low_memory=False)


# ----------- Daten sind bereits komplett (inkl. Temperatur)
df_full = df_prep
print(f"Daten erfolgreich aus 3.1_Preprocessing geladen! Shape: {df_full.shape}")


Verwendeter Preprocessing-Ordner: 2025-12-23_13-57-18
Daten erfolgreich aus 3.1_Preprocessing geladen! Shape: (175099, 23)


<div class="alert alert-info">
    <b>IntraSOM Machine Learning</b><br><br>
    <b>Datenquelle:</b><br>
    - Preprocessed Data: Ionen (Log + Quantile Gaussian) + Temperatur (Cleaned)<br>
    <br>
    <b>Filter:</b><br>
    - <b>Nur Temperaturen > 10 °C</b> werden berücksichtigt.
    <br>
    <b>Library:</b><br>
    - <a href="https://github.com/InTRA-USP/IntraSOM">IntraSOM</a>
</div>

In [3]:
# ----------- Features auswählen
target_ions = ["Na", "Ca", "Mg", "Cl", "HCO3", "SO4"]
features = []
for col in df_full.columns:
    if "_log_gauss" in col:
        element = col.split("_")[0]
        if element in target_ions:
            features.append(col)

print(f"Input Features für SOM: {features}")

# ----------- Filtern
df_som = df_full[features + ['temperature_in_c']].copy()

# ---------- WICHTIG: Temperatur in Numeric wandeln (Fehler abfangen)
df_som['temperature_in_c'] = pd.to_numeric(df_som['temperature_in_c'], errors='coerce')

# ---------- FILTER: Nur Temperaturen > 10 Grad
len_before_temp = len(df_som)
df_som = df_som[df_som['temperature_in_c'] > 10]
print(f"Filter Temp > 10°C: {len_before_temp - len(df_som)} Zeilen entfernt. Verbleibend: {len(df_som)}")

initial_len = len(df_som)
df_som.dropna(subset=features, inplace=True)
print(f"Zeilen mit fehlenden Features entfernt: {initial_len - len(df_som)}. Final: {len(df_som)}")

Input Features für SOM: ['Na_in_mmol/L_log_gauss', 'Mg_in_mmol/L_log_gauss', 'Ca_in_mmol/L_log_gauss', 'Cl_in_mmol/L_log_gauss', 'SO4_in_mmol/L_log_gauss', 'HCO3_in_mmol/L_log_gauss']
Filter Temp > 10°C: 130898 Zeilen entfernt. Verbleibend: 44201
Zeilen mit fehlenden Features entfernt: 37411. Final: 6790


In [4]:
# ----------- IntraSOM erwartet Input Data
# Da unsere Features bereits _log_gauss (normalverteilt) transformiert sind, 
# müssen wir sie nicht zwingend auf [0,1] MinMax scalen, wie es MiniSom manchmal mag.
# IntraSOM kann 'normalization' Parameter selbst handeln (z.B. 'var' = Varianz).
# Wir übergeben die Daten vorerst wie sie sind (Quasi-Standard-Normalverteilung),
# da sie durch QuantileTransformer schon skaliert wurden.

data_values = df_som[features].values
print(f"Data Shape für IntraSOM: {data_values.shape}")

Data Shape für IntraSOM: (6790, 6)


In [5]:
# ----------- IntraSOM Training
if 'som' in locals():
    del som # Reset falls cell re-run

# Map Config
mapsize = (10, 10)

# Build SOM
print("Building IntraSOM model...")
som = intrasom.SOMFactory.build(
    data=data_values,
    mapsize=mapsize,
    mapshape='toroid',    # Wie im Github Example
    lattice='hexa',       # Hexagonal gewünscht
    normalization=None,   # Features sind schon preprocessed (Mean~0, Std~1)
    initialization='random',
    neighborhood='gaussian',
    training='batch',
    name='IntraSOM_Analysis',
    component_names=features,
    missing=False         # Wir haben NAs vorher entfernt
)

print("Starting training...")
som.train()

Building IntraSOM model...
Loading dataframe...
Normalizing data...
Creating neighborhood...
Initializing map...


Creating Neuron Distance Rows:   0%|          | 0/10 [00:00<?, ?rows/s]

Starting training...
Starting Training...
Rough Training:


  0%|          | 0/1 [00:00<?, ?it/s]

Fine Tuning:


  0%|          | 0/1 [00:00<?, ?it/s]

Saving...
Training Report Created
Training completed successfully.


In [6]:
# ----------- Visualization Output Folder
output_root = base_dir / "IntraSom_Results"
output_root.mkdir(exist_ok=True)

# ----------- Visualization
# IntraSOM bietet eigene Plotting Tools via PlotFactory
print("Training complete. Generating visualizations...")

plot_factory = intrasom.PlotFactory(som)

# 1. U-Matrix
print("Plotting U-Matrix...")
plot_factory.plot_umatrix(
    hits=True, 
    title="U-Matrix (Distance Map) - Hexagonal"
)
# Speichern des aktuellen Plots (da IntraSOM das ggf. nur anzeigt)
timestamp = datetime.now().strftime("%H-%M-%S")
plt.savefig(output_root / f"IntraSOM_UMatrix_{timestamp}.png")
plt.show()

print(f"Results should be saved in: {output_root}")

Training complete. Generating visualizations...


AttributeError: module 'intrasom' has no attribute 'PlotFactory'